In [ ]:
def extract_features(number):
    transcript = pd.read_csv(os.path.join(prefix, 'DAIC/{0}_P/{0}_TRANSCRIPT.csv'.format(number)), sep='\t').fillna('')
    
    wavefile = wave.open(os.path.join(prefix, 'DAIC/{0}_P/{0}_AUDIO.wav'.format(number, 'r')))
    sr = wavefile.getframerate()
    nframes = wavefile.getnframes()
    wave_data = np.frombuffer(wavefile.readframes(nframes), dtype=np.short)
    
    response = ''
    start_time = 0
    stop_time = 0
    feats = []
    signal = []

    for t in transcript.itertuples():
        # 问题开始
        if getattr(t,'speaker') == 'Ellie' and (identify_topics(getattr(t,'value')) or 'i think i have asked everything' in getattr(t,'value')):
            # 初始化
            response = ''
            if len(signal) == 0:
                continue
            feats.append(wav2vlad(signal, sr))
            signal = []
        elif getattr(t,'speaker') == 'Participant':
            if 'scrubbed_entry' in getattr(t,'value'):
                continue
            start_time = int(getattr(t,'start_time')*sr)
            stop_time = int(getattr(t,'stop_time')*sr)
            response += (' ' + getattr(t,'value'))
            signal = np.hstack((signal, wave_data[start_time:stop_time].astype(np.float)))
    
    print(np.shape(feats))
    print('{}_P feature done'.format(number))
    return feats

In [ ]:
from pydub import AudioSegment
from os import listdir
from os.path import isfile, join

def get_silence(audio, threshold, interval):
    #"get length of silence in seconds from a wav file"

    # swap out pydub import for other types of audio
    song = AudioSegment.from_wav(audio)

    # break into chunks
    chunks = [song[i:i+interval] for i in range(0, len(song), interval)]

    # find number of chunks with dBFS below threshold
    silent_blocks = 0
    for c in chunks:
        if c.dBFS == float('-inf') or c.dBFS < threshold:
            silent_blocks += 1
        else:
            break

    # convert blocks into seconds
    return round(silent_blocks * (interval/1000), 3)

# get files in a directory
#audio_path = 'path/to/directory'
audio_path = '431_AUDIO.wav'
#audio_files = [i for i in listdir(audio_path) if isfile(join(audio_path, i))]

threshold = -30 # tweak based on signal-to-noise ratio

interval = 1 # ms, increase to speed up
silence=get_silence(audio_path, threshold, interval)
print(silence)
#leading_silences = {a: get_silence(join(audio_path, a),
#                                   threshold, interval) for a in audio_files}

## to get tab-separated values:
#for name, leading_silence in leading_silences.items():
  #  print(''.join([name, '\t', str(leading_silence)]))

In [ ]:
import wave
import matplotlib.pyplot as plt
import numpy as np
import os
f = wave.open(r"300_AUDIO.wav",'rb')
params = f.getparams()
nchannels ,sampwidth ,framerate ,nframes = params [:4]
#print(framerate)

strData = f.readframes(nframes)
waveDate = np.frombuffer(strData,dtype=np.int16)

waveDate = waveDate*1.0/(max(abs(waveDate)))

time = np.arange(0,nframes)*(1.0/framerate)

#plt.plot(time,waveDate)
#plt.xlabel("Time")
#plt.ylabel("Amplitude")
#plt.title("Single channel wavedata")
#plt.show()

print(max(waveDate))
print(min(waveDate))
print(np.mean(waveDate))

In [ ]:
import librosa
#x , sr = librosa.load("negative.wav", sr=8000)
#print(x.shape, sr)
import matplotlib.pyplot as plt
import librosa.display
#plt.figure(figsize=(14, 5))
#x = wave.open(r"301_AUDIO.wav",'rb')
x=waveDate
sr=8000
#librosa.display.waveplot(x, sr=sr)

X = librosa.stft(x)
Xdb = librosa.amplitude_to_db(abs(X))   
plt.figure(figsize=(14, 5))
librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar()
print(Xdb.shape)

In [ ]:
n0 = 9000
n1 = 9100
#plt.figure(figsize=(14, 5))
#plt.plot(x[n0:n1])
#plt.grid()
zero_crossings = librosa.zero_crossings(x[n0:n1], pad=False)
print(sum(zero_crossings))

In [ ]:
audio, sr = librosa.load('300_AUDIO.wav')#audio path
D = librosa.stft(audio, n_fft=2048, hop_length=512)
magnitude, phase = librosa.magphase(D)
cent = librosa.feature.spectral_centroid(y=audio, sr=sr)
plt.figure(figsize=(15,10))
ax1 = plt.subplot(2,1,1)
librosa.display.waveshow(audio, sr=sr, alpha=0.8)
plt.title('audio data')
ax2 = plt.subplot(2,1,2, sharex=ax1)
librosa.display.specshow(librosa.amplitude_to_db(magnitude, ref=np.max), sr=sr, y_axis='linear', x_axis='time')
#print(np.mean(librosa.amplitude_to_db(magnitude, ref=np.max)))
cent_times = librosa.frames_to_time(np.arange(len(cent[0])), sr=sr, hop_length=512)
plt.plot(cent_times, cent[0], label='Spectral Centroid', lw=4, color='orange')
plt.legend()
plt.title('STFT spectrogram & Spectral Centroid')
print(np.mean(cent_times))